# 02 · Silver · `atendimentos`
### V-Energy Grid 360 — Arquitetura Medalhão


Lê `v_energy.bronze.atendimentos` (`atendimentos.json`; valores simples como texto e `historico` como lista de registros) e grava **duas** tabelas:
- `v_energy.silver.atendimentos`: um chamado por linha, chave `protocolo`;
- `v_energy.silver.tratativas_atendimento`: uma tratativa por linha, chave `(protocolo, sequencia)`.

**Fonte das regras:** `exploracao_atendimentos.ipynb` e a seção `atendimentos` do `docs/data_quality_log.md` (itens A1 a A7). As decisões foram tomadas pela responsável pela tabela, com a autonomia combinada com o grupo, e estão registradas no log com as alternativas consideradas.

**Dependência:** `silver.unidades_consumidoras` (Aimê) precisa existir, porque uma checagem confere o `id_uc` contra ela. No Workflow, este notebook roda depois de `02_silver_unidades_consumidoras`.

**`atendimentos` é a tabela mais limpa do projeto.** A exploração não encontrou cópias, órfãs, cabeçalho repetido, datas misturadas nem sentinelas. Por isso **não há tratamento** para cabeçalho repetido, cópias ou UC inexistente: esses casos ficam só como **checagens** (seção 10), e, se aparecerem numa carga futura, a Silver não é gravada e o grupo decide, em vez de limpar em silêncio.

| # | Tratamento | Linhas afetadas | Regra |
|---|---|---|---|
| 1 | Datas em texto, em dois formatos (chamado com espaço, tratativa com `T`) e sentinela `9999` (A4, A7) | 318.871 chamados e 636.857 tratativas | `try_to_timestamp` com o formato de cada coluna; sentinela vira nulo |
| 2 | `nota_satisfacao` nula, chamado não avaliado (A1) | 9.566 (3,0%) | **manter nula**; `try_cast` para `decimal(3,1)` |
| 3 | Categorias (`canal`, `assunto`, `subassunto`, `acao`) | 0 variações | `upper(trim)` (proteção) |
| 4 | `historico`: lista aninhada (A3) | 318.871 chamados e 636.857 tratativas | `posexplode_outer` em `silver.tratativas_atendimento` |
| 5 | Colunas derivadas | todas | `mes_referencia`, `aberto`, `tempo_resolucao_horas`, `qtd_tratativas` |
| 6 | Pico de julho de 2025: 18.061 chamados, 2,8× o normal (A6) | ~11.660 acima do normal | **manter tudo** e documentar |

**Ordem das operações:** converter datas e nota, padronizar categorias, derivar as colunas do chamado e só então abrir o `historico`, a partir do chamado já tratado, para as duas tabelas saírem do mesmo conjunto de chamados.

**Resultado esperado:** 318.871 chamados (`protocolo` único) e 636.857 tratativas, 1 a 3 por chamado.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_bronze = f"{bronze_schema}.atendimentos"
tabela_uc = f"{silver_schema}.unidades_consumidoras"      # cadastro de referência para validar o id_uc
tabela_silver = f"{silver_schema}.atendimentos"
tabela_silver_trat = f"{silver_schema}.tratativas_atendimento"

print(f"origem:   {tabela_bronze}")
print(f"cadastro: {tabela_uc}")
print(f"destinos: {tabela_silver} e {tabela_silver_trat}")

# A Silver de UCs precisa existir. Se o nome for outro no seu workspace, ajuste tabela_uc acima.
try:
    n_ucs = spark.table(tabela_uc).select("id_uc").distinct().count()
except Exception as e:
    raise Exception(f"{tabela_uc} não encontrada. Rode 02_silver_unidades_consumidoras antes, ou ajuste tabela_uc.") from e
print("UCs no cadastro (Silver):", f"{n_ucs:,}".replace(",", "."))   # esperado: 45.000

origem:   v_energy.bronze.atendimentos
cadastro: v_energy.silver.unidades_consumidoras
destinos: v_energy.silver.atendimentos e v_energy.silver.tratativas_atendimento
UCs no cadastro (Silver): 45.000


## 2. Funções de checagem de qualidade

Mesmas funções da aula `02_Bronze_to_Silver`, para que as Silvers do grupo gravem em `silver.dq_log` com as mesmas colunas (`table_name`, `check_name`, `total_rows`, `failed_rows`, `passed`, `checked_at`).

- `dq_check(table_name, check_name, df, condition)`: `condition` é a condição **esperada** (o que deveria ser verdade em todas as linhas). Falham as linhas que a violam.
- `dq_check_unique(table_name, check_name, df, key_cols)`: passa se a chave não se repete.
- `dq_check_contagem(table_name, check_name, df, esperado)`: passa só se o número de linhas for exatamente o esperado.

As funções só acumulam os resultados na lista `dq_results`. Depois das checagens, o notebook faz duas coisas, nesta ordem: **grava o resultado em `silver.dq_log`** (sempre, inclusive quando há falha, porque o registro da falha também é útil) e **só então decide se grava as Silvers** (só se todas as checagens, das duas tabelas, passarem).

**Cuidado com nulos:** linhas em que a condição resulta em nulo não contam como falha (o filtro `~condition` descarta nulos). Quando o nulo também for um problema, escreva um `dq_check` separado com `isNotNull()` ou use `F.coalesce(condição, F.lit(False))`.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

## 3. Leitura da Bronze

Leitura da tabela Bronze, sem transformação nenhuma. Conferências antes de começar:
- **318.871** chamados.
- Os valores simples (`protocolo`, `id_uc`, `canal`, `assunto`, `subassunto`, `data_abertura`, `data_fechamento`, `nota_satisfacao`) devem aparecer como `string`: é a regra da Bronze (`primitivesAsString`). A exceção é o `historico`, uma lista de registros (`acao`, `canal`, `data`, `sequencia`, também texto), e a coluna de auditoria `ingestion_timestamp`.
- **636.857** tratativas somadas no `historico`.

O DataFrame `bronze` fica guardado para a comparação Bronze × Silver no fim do notebook.

In [0]:
bronze = spark.table(tabela_bronze)

n_bronze = bronze.count()
n_trat_bronze = bronze.agg(F.sum(F.size("historico"))).first()[0]
print(f"Chamados na Bronze:   {n_br(n_bronze)}")        # esperado: 318.871
print(f"Tratativas na Bronze: {n_br(n_trat_bronze)}")   # esperado: 636.857

bronze.printSchema()
display(bronze.limit(5))

Chamados na Bronze:   318.871
Tratativas na Bronze: 636.857
root
 |-- assunto: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- data_abertura: string (nullable = true)
 |-- data_fechamento: string (nullable = true)
 |-- historico: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- acao: string (nullable = true)
 |    |    |-- canal: string (nullable = true)
 |    |    |-- data: string (nullable = true)
 |    |    |-- sequencia: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- nota_satisfacao: string (nullable = true)
 |-- protocolo: string (nullable = true)
 |-- subassunto: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



assunto,canal,data_abertura,data_fechamento,historico,id_uc,nota_satisfacao,protocolo,subassunto,source_file,ingestion_timestamp
RELIGACAO,SITE,2022-07-02 15:51:00,2022-07-10 14:51:00,"List(List(RETORNO AO CLIENTE, SITE, 2022-07-10T14:51:00, 1))",UC142511,3.9,AT000000000,URGENCIA,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z
FALTA DE ENERGIA,SITE,2022-07-07 23:25:00,2022-07-17 10:25:00,"List(List(RETORNO AO CLIENTE, SITE, 2022-07-12T16:25:00, 1), List(DESPACHO, SITE, 2022-07-17T10:25:00, 2))",UC100661,4.1,AT000000001,INTERRUPCAO TOTAL,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z
LIGACAO NOVA,WHATSAPP,2022-07-11 10:48:00,2022-07-13 21:48:00,"List(List(RETORNO AO CLIENTE, WHATSAPP, 2022-07-12T05:48:00, 1), List(ABERTURA, WHATSAPP, 2022-07-13T01:48:00, 2), List(ENCERRAMENTO, WHATSAPP, 2022-07-13T21:48:00, 3))",UC108189,4.4,AT000000002,SOLICITACAO,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z
DANO ELETRICO,TELEFONE,2022-07-09 17:11:00,2022-07-16 05:11:00,"List(List(ANALISE, TELEFONE, 2022-07-11T21:11:00, 1), List(ABERTURA, TELEFONE, 2022-07-14T01:11:00, 2), List(RETORNO AO CLIENTE, TELEFONE, 2022-07-16T05:11:00, 3))",UC104975,4.0,AT000000003,VISTORIA,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z
FATURA,WHATSAPP,2022-07-23 15:44:00,2022-07-27 19:44:00,"List(List(ABERTURA, WHATSAPP, 2022-07-25T17:44:00, 1), List(ENCERRAMENTO, WHATSAPP, 2022-07-27T19:44:00, 2))",UC108340,4.0,AT000000004,SEGUNDA VIA,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z


## 4. Datas: `data_abertura`, `data_fechamento` e a data de cada tratativa (log A4 e A7)

**Decisão:** converter para `timestamp` com `try_to_timestamp`, usando o formato de cada coluna:

| Coluna | Formato (texto na Bronze) | Linhas |
|---|---|---|
| `data_abertura`, `data_fechamento` | `yyyy-MM-dd HH:mm:ss` (espaço entre data e hora) | 318.871 cada |
| `historico.data` (a de cada tratativa) | `yyyy-MM-dd'T'HH:mm:ss` (com `T`) | 636.857 |

**Evidência:** a exploração encontrou **formato único** em cada coluna, sem sentinela (nenhum valor começa com `9999`). Como só existe o padrão ISO, não há ordem dia/mês a descobrir, e o risco que existia em `leituras` e `faturas` (hífen como mês primeiro) não aparece aqui. Mesmo assim, a conversão foi conferida contra um gabarito independente, o histórico do próprio chamado: `data_abertura <= primeira tratativa` e `data_fechamento == última tratativa` valem em **318.871 de 318.871**, sem nenhuma conversão falhando. Essas duas regras viram checagens na seção 10.

**Por que `timestamp` e não `date`:** a hora é necessária para o `tempo_resolucao_horas` (seção 7). A Gold converte para `date` onde precisar.

**Sentinela:** por convenção do grupo (`9999-12-31` vira nulo), qualquer `data_fechamento` que comece com `9999` vira nulo, o que significa chamado aberto. Hoje são 0.

**Aspas em SQL:** no formato da tratativa, o `T` vai entre aspas simples. Em SQL do Spark, `''` não escapa aspa (junta dois textos), então a aspa é escrita como `\'` (a função `sql_fmt` abaixo faz isso).

**Alternativa considerada:** `try_cast(... as timestamp)`, que aceita o padrão ISO sem formato explícito. Descartada como regra porque aceitaria em silêncio um formato novo; fica só como conferência cruzada (a célula abaixo imprime quantas conversões divergem, esperado 0).

**Conferência:** nenhuma `data_abertura` nula depois da conversão e nenhuma `data_fechamento` preenchida que não converteu.

In [0]:
# Tratamento 1 · datas (esperado: 1 padrão por coluna; 0 não convertidas; 0 divergências com try_cast)
FMT_CHAMADO = "yyyy-MM-dd HH:mm:ss"
FMT_TRATATIVA = "yyyy-MM-dd'T'HH:mm:ss"

def sql_fmt(fmt):
    """Em SQL do Spark, '' não escapa aspa: usar \\' dentro do literal."""
    return fmt.replace(chr(39), chr(92) + chr(39))

def ts(coluna, fmt):
    return F.expr(f"try_to_timestamp({coluna}, '{sql_fmt(fmt)}')")

# padrões presentes (esperado: só 9999-99-99 99:99:99, 318.871 linhas em cada coluna)
for c in ["data_abertura", "data_fechamento"]:
    print(c)
    (bronze.withColumn("padrao", F.regexp_replace(c, "[0-9]", "9"))
               .groupBy("padrao").count().orderBy(F.desc("count")).show(truncate=False))

# conferência cruzada: try_to_timestamp com formato x try_cast (esperado: 0 divergências)
cruz = bronze.select(ts("data_abertura", FMT_CHAMADO).alias("a"),
                         F.expr("try_cast(data_abertura as timestamp)").alias("b"))
print("data_abertura: try_to_timestamp x try_cast divergentes:", n_br(cruz.filter(~F.col("a").eqNullSafe(F.col("b"))).count()))

df_dt = (
    bronze
    .withColumn("fechamento_txt", F.col("data_fechamento"))     # auxiliar: texto original, para uma checagem
    .withColumn("data_abertura", ts("data_abertura", FMT_CHAMADO))
    .withColumn(
        "data_fechamento",
        F.when(F.col("fechamento_txt").startswith("9999"), F.lit(None).cast("timestamp"))   # sentinela vira nulo
         .otherwise(ts("data_fechamento", FMT_CHAMADO)),
    )
)

print("data_abertura nula após a conversão:", n_br(df_dt.filter("data_abertura IS NULL").count()))                      # esperado: 0
print("data_fechamento preenchida e não convertida:",
      n_br(df_dt.filter("fechamento_txt IS NOT NULL AND NOT startswith(fechamento_txt, '9999') AND data_fechamento IS NULL").count()))  # esperado: 0
print("data_fechamento nula (chamados abertos):", n_br(df_dt.filter("data_fechamento IS NULL").count()))                # esperado: 0
df_dt.agg(F.min("data_abertura").alias("abertura_min"), F.max("data_abertura").alias("abertura_max"),
          F.max("data_fechamento").alias("fechamento_max")).show(truncate=False)
# esperado: aberturas de 2022-07-01 a 2026-06-30; fechamentos até 2026-07-10 (chamados abertos no fim de junho)

data_abertura
+-------------------+------+
|padrao             |count |
+-------------------+------+
|9999-99-99 99:99:99|318871|
+-------------------+------+

data_fechamento
+-------------------+------+
|padrao             |count |
+-------------------+------+
|9999-99-99 99:99:99|318871|
+-------------------+------+

data_abertura: try_to_timestamp x try_cast divergentes: 0
data_abertura nula após a conversão: 0
data_fechamento preenchida e não convertida: 0
data_fechamento nula (chamados abertos): 0
+-------------------+-------------------+-------------------+
|abertura_min       |abertura_max       |fechamento_max     |
+-------------------+-------------------+-------------------+
|2022-07-01 00:00:00|2026-06-30 23:56:00|2026-07-10 14:27:00|
+-------------------+-------------------+-------------------+



## 5. Nota de satisfação (log A1 e A2)

**Decisão:** converter para `decimal(3,1)` com `try_cast` e **manter nulo** onde a nota não foi preenchida. Uma vírgula decimal, se aparecesse, seria trocada por ponto antes do cast (proteção; não há nenhuma).

**Evidência:**
- 309.305 notas preenchidas e **9.566 nulas (3,0%)**. Todas as nulas são de chamados fechados (nenhum chamado está aberto), então o nulo significa "o cliente não avaliou": **nulo de negócio**, que tem significado e deve ser preservado. Preencher com a média ou com zero inventaria uma avaliação.
- Nenhuma nota é texto e nenhuma tem vírgula. A faixa vai de 1,3 a 5,0, com média 4,043.
- **No máximo 1 casa decimal** na origem (conferido na execução da Silver), por isso `decimal(3,1)`: o cast não arredonda nada. A checagem da seção 10 mantém essa garantia.
- Há um acúmulo no teto: 6.160 chamados com 5,0 contra 4.363 com 4,9, quando a curva já descia. É compatível com notas acima de 5 cortadas em 5,0 (**hipótese**, sem ação). A nota é mantida como veio e a hipótese vai para o dicionário de dados.

**Impacto na Gold:** `nota_satisfacao_media` deve ser a média das **notas preenchidas** (a média do Spark já ignora nulos). O contrato da Gold marca "escala a confirmar": a escala observada é de 1 a 5.

**Conferência:** 9.566 nulas depois do cast (as mesmas da origem) e nenhuma nota preenchida que deixou de converter.

In [0]:
# Tratamento 2 · nota de satisfação (esperado: 9.566 nulas; 0 preenchidas que não convertem)
df_nota = (
    df_dt
    .withColumn("nota_txt", F.regexp_replace(F.trim(F.col("nota_satisfacao")), ",", "."))   # auxiliar: texto normalizado
    .withColumn("nota_satisfacao", F.expr("try_cast(nota_txt as decimal(3,1))"))
)

print("nota nula:", n_br(df_nota.filter("nota_satisfacao IS NULL").count()))                                       # esperado: 9.566
print("nota preenchida que não converteu:", n_br(df_nota.filter("nota_txt IS NOT NULL AND nota_satisfacao IS NULL").count()))  # esperado: 0
print("nota com vírgula na origem:", n_br(df_dt.filter(F.col("nota_satisfacao").contains(",")).count()))   # esperado: 0
# casas decimais máximas nas notas de origem (esperado: 1, confirmado na primeira execução)
df_nota.agg(F.max(F.length(F.regexp_extract("nota_txt", r"\.([0-9]+)$", 1))).alias("max_casas_decimais")).show()
# esperado: mínimo 1.3, máximo 5.0, média 4.043
df_nota.agg(F.min("nota_satisfacao").alias("min"), F.max("nota_satisfacao").alias("max"),
            F.round(F.avg(F.col("nota_satisfacao").cast("double")), 3).alias("media")).show()
# esperado: 6.160 notas em 5,0 (acúmulo no teto) e 4.363 em 4,9 (arredondando a 1 casa)
(df_nota.filter("nota_satisfacao IS NOT NULL")
        .withColumn("nota_1_casa", F.round(F.col("nota_satisfacao").cast("double"), 1))
        .filter(F.col("nota_1_casa").isin(4.9, 5.0))
        .groupBy("nota_1_casa").count().orderBy("nota_1_casa").show())

nota nula: 9.566
nota preenchida que não converteu: 0
nota com vírgula na origem: 0
+------------------+
|max_casas_decimais|
+------------------+
|                 1|
+------------------+

+---+---+-----+
|min|max|media|
+---+---+-----+
|1.3|5.0|4.043|
+---+---+-----+

+-----------+-----+
|nota_1_casa|count|
+-----------+-----+
|        4.9| 4363|
|        5.0| 6160|
+-----------+-----+



## 6. Padronização das categorias

**Decisão:** `upper(trim)` em `canal`, `assunto` e `subassunto`, como proteção.

**Evidência:** sem variações de grafia. 5 canais (TELEFONE, APP, WHATSAPP, SITE e PRESENCIAL, ~64 mil cada), 6 assuntos (`FALTA DE ENERGIA` concentra 31,8% dos chamados e os outros cinco têm ~13,6% cada) e 15 subassuntos, formando 17 pares assunto × subassunto, todos coerentes (cada subassunto aparece sob um único assunto, exceto `VISTORIA`, que aparece sob três: `DANO ELETRICO`, `GERACAO DISTRIBUIDA` e `LIGACAO NOVA`). Os valores já vêm em maiúsculas e sem acento, e a Gold e o agente de IA vão exibi-los assim.

**Por que os domínios viram checagem:** se uma carga futura trouxer um canal ou assunto novo, ou uma grafia diferente, a checagem de domínio falha e avisa, em vez de criar uma categoria nova sem ninguém perceber.

**Conferência:** 5 canais, 6 assuntos, 15 subassuntos e 17 pares depois do `upper(trim)`.

In [0]:
# Tratamento 3 · categorias (esperado: 5 canais, 6 assuntos, 15 subassuntos, 17 pares)
df_cat = (
    df_nota
    .withColumn("canal", F.upper(F.trim("canal")))
    .withColumn("assunto", F.upper(F.trim("assunto")))
    .withColumn("subassunto", F.upper(F.trim("subassunto")))
)

print("canais distintos:      ", df_cat.select("canal").distinct().count())                           # esperado: 5
print("assuntos distintos:    ", df_cat.select("assunto").distinct().count())                         # esperado: 6
print("subassuntos distintos: ", df_cat.select("subassunto").distinct().count())                      # esperado: 15
print("pares assunto x subassunto:", df_cat.select("assunto", "subassunto").distinct().count())       # esperado: 17
df_cat.groupBy("assunto").count().orderBy(F.desc("count")).show(truncate=False)                       # FALTA DE ENERGIA 101.306; demais de 43.340 a 43.624

canais distintos:       5
assuntos distintos:     6
subassuntos distintos:  15
pares assunto x subassunto: 17
+-------------------+------+
|assunto            |count |
+-------------------+------+
|FALTA DE ENERGIA   |101306|
|FATURA             |43624 |
|DANO ELETRICO      |43582 |
|LIGACAO NOVA       |43520 |
|GERACAO DISTRIBUIDA|43499 |
|RELIGACAO          |43340 |
+-------------------+------+



## 7. Colunas derivadas do chamado (log A5 e A7)

**Decisão:** derivar quatro colunas, todas a partir do que já foi convertido:

| Coluna | Regra | Para quê |
|---|---|---|
| `mes_referencia` | `yyyy-MM` da `data_abertura` | mesmo formato das outras Silvers; `kpis_mensal` conta chamados por mês |
| `aberto` | `data_fechamento` nula | chamados em aberto da visão 360; **constante falsa hoje** (nenhum chamado está aberto), mas protege cargas futuras |
| `tempo_resolucao_horas` | `(fechamento − abertura)` em horas, 2 casas | qualidade do serviço; nulo se o chamado está aberto |
| `qtd_tratativas` | tamanho do `historico` | de 1 a 3 em todos os chamados |

**O que não se deriva: o status a partir da última ação.** A exploração mostrou que a última ação **não indica o status**: todos os chamados estão fechados, mas a última ação se divide entre as cinco (DESPACHO 64.257, RETORNO AO CLIENTE 63.987, ABERTURA 63.903, ENCERRAMENTO 63.432 e ANALISE 63.292). O histórico é uma amostra de ações, e não um processo completo. Por isso o estado do chamado vem de `aberto` (a data de fechamento), e a `ultima_acao` da Gold, se for mantida, deve ser lida como "última ação registrada", e não como situação.

**Impacto na Gold:** `fato_atendimento` usa `mes_referencia`, `aberto`, `tempo_resolucao_horas` e `qtd_tratativas` direto desta tabela. `ultima_acao` sai da maior `sequencia` de `silver.tratativas_atendimento`.

**Conferência:** `qtd_tratativas` entre 1 e 3 (106.717 chamados com 1, 106.322 com 2 e 105.832 com 3) e `aberto` falso em todos.

In [0]:
# Colunas derivadas (esperado: aberto falso em todos; qtd_tratativas 1/2/3 = 106.717 / 106.322 / 105.832)
df_der = (
    df_cat
    .withColumn("mes_referencia", F.date_format("data_abertura", "yyyy-MM"))
    .withColumn("aberto", F.col("data_fechamento").isNull())
    .withColumn(
        "tempo_resolucao_horas",
        F.round((F.col("data_fechamento").cast("long") - F.col("data_abertura").cast("long")) / 3600, 2).cast("decimal(10,2)"),
    )
    .withColumn("qtd_tratativas", F.when(F.col("historico").isNull(), 0).otherwise(F.size("historico")))
)

df_der.groupBy("aberto").count().show()
df_der.groupBy("qtd_tratativas").count().orderBy("qtd_tratativas").show()
df_der.agg(F.min("tempo_resolucao_horas").alias("min_h"),
           F.round(F.avg(F.col("tempo_resolucao_horas").cast("double")), 1).alias("media_h"),
           F.max("tempo_resolucao_horas").alias("max_h")).show()
# o mínimo deve ser >= 0; a média geral não consta da exploração (julho/2025: 361,1 h; demais meses: 119,9 h)

+------+------+
|aberto| count|
+------+------+
| false|318871|
+------+------+

+--------------+------+
|qtd_tratativas| count|
+--------------+------+
|             1|106717|
|             2|106322|
|             3|105832|
+--------------+------+

+-----+-------+------+
|min_h|media_h| max_h|
+-----+-------+------+
| 1.00|  133.6|719.00|
+-----+-------+------+



## 8. O `historico` vira uma tabela: `silver.tratativas_atendimento` (log A3)

**Decisão:** abrir o `historico` com `posexplode_outer` em uma tabela própria, com uma linha por tratativa e chave `(protocolo, sequencia)`. Colunas: `protocolo`, `sequencia` (inteiro), `acao`, `canal`, `data_tratativa` (`timestamp`) e `source_file`. A lista não vai para a Silver do chamado: ela continua só na Bronze, que preserva a origem.

**Evidência:** 636.857 tratativas, de 1 a 3 por chamado, em partes iguais. Na exploração:
- `sequencia` sempre vira número e é igual à posição na lista + 1 (0 exceções);
- as datas das tratativas estão em ordem crescente dentro de cada chamado (0 fora de ordem);
- nenhuma tratativa fica antes da abertura nem depois do fechamento do chamado;
- 5 ações (ABERTURA 127.796, RETORNO AO CLIENTE 127.484, DESPACHO 127.417, ENCERRAMENTO 127.256 e ANALISE 126.904) e os mesmos 5 canais do chamado, sem variação de grafia.

**Detalhes:**
- A tabela sai do `df_der` (chamados já tratados), então as duas Silvers têm exatamente os mesmos protocolos.
- `acao` e `canal` com `upper(trim)`, como proteção.
- `pos` (posição na lista, a partir de 0) só serve para checar a `sequencia`, e não vai para a Silver.
- `posexplode_outer` mantém chamados com `historico` vazio ou nulo; o filtro `t IS NOT NULL` os tira da tabela de tratativas, e a checagem de `qtd_tratativas` entre 1 e 3 acusa o chamado. Hoje são 0.

**Alternativa considerada:** guardar `historico` como lista na Silver do chamado. Descartada porque a Gold, o portal (SQLite ou PostgreSQL) e o agente de SQL não leem listas, e a abertura em tabela era o que a Bronze deixou combinado (`AGENTS.md`: abertura só na Silver).

**Conferência:** 636.857 tratativas; nenhuma `sequencia` nem data que não converteu.

In [0]:
# Tratamento 4 · abrir o historico (esperado: 636.857 tratativas; 0 sequencias e 0 datas sem converter)
df_trat_full = (
    df_der
    .select("protocolo", "source_file", F.posexplode_outer("historico").alias("pos", "t"))
    .filter(F.col("t").isNotNull())
    .select(
        "protocolo", "source_file", "pos",
        F.col("t.sequencia").alias("sequencia_txt"),
        F.upper(F.trim(F.col("t.acao"))).alias("acao"),
        F.upper(F.trim(F.col("t.canal"))).alias("canal"),
        F.col("t.data").alias("data_txt"),
    )
    .withColumn("sequencia", F.expr("try_cast(trim(sequencia_txt) as int)"))
    .withColumn("data_tratativa", ts("data_txt", FMT_TRATATIVA))
)

n_trat = df_trat_full.count()
print(f"Tratativas: {n_br(n_trat)}")                                                                   # esperado: 636.857
print("sequencia que não converteu:", n_br(df_trat_full.filter("sequencia IS NULL").count()))          # esperado: 0
print("data da tratativa que não converteu:", n_br(df_trat_full.filter("data_tratativa IS NULL").count()))  # esperado: 0

# padrão da data da tratativa (esperado: só 9999-99-99T99:99:99, 636.857 linhas)
(df_trat_full.withColumn("padrao", F.regexp_replace("data_txt", "[0-9]", "9"))
             .groupBy("padrao").count().show(truncate=False))

# conferência cruzada: try_to_timestamp com formato x try_cast (esperado: 0 divergências)
cruz_t = df_trat_full.select(F.col("data_tratativa").alias("a"), F.expr("try_cast(data_txt as timestamp)").alias("b"))
print("data da tratativa: try_to_timestamp x try_cast divergentes:", n_br(cruz_t.filter(~F.col("a").eqNullSafe(F.col("b"))).count()))

df_trat_full.groupBy("acao").count().orderBy(F.desc("count")).show(truncate=False)    # 127.796 / 127.484 / 127.417 / 127.256 / 126.904
display(df_trat_full.limit(5))

Tratativas: 636.857
sequencia que não converteu: 0
data da tratativa que não converteu: 0
+-------------------+------+
|padrao             |count |
+-------------------+------+
|9999-99-99T99:99:99|636857|
+-------------------+------+

data da tratativa: try_to_timestamp x try_cast divergentes: 0
+------------------+------+
|acao              |count |
+------------------+------+
|ABERTURA          |127796|
|RETORNO AO CLIENTE|127484|
|DESPACHO          |127417|
|ENCERRAMENTO      |127256|
|ANALISE           |126904|
+------------------+------+



protocolo,source_file,pos,sequencia_txt,acao,canal,data_txt,sequencia,data_tratativa
AT000000000,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,0,1,RETORNO AO CLIENTE,SITE,2022-07-10T14:51:00,1,2022-07-10T14:51:00.000Z
AT000000001,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,0,1,RETORNO AO CLIENTE,SITE,2022-07-12T16:25:00,1,2022-07-12T16:25:00.000Z
AT000000001,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,1,2,DESPACHO,SITE,2022-07-17T10:25:00,2,2022-07-17T10:25:00.000Z
AT000000002,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,0,1,RETORNO AO CLIENTE,WHATSAPP,2022-07-12T05:48:00,1,2022-07-12T05:48:00.000Z
AT000000002,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,1,2,ABERTURA,WHATSAPP,2022-07-13T01:48:00,2,2022-07-13T01:48:00.000Z


## 9. Seleção final das colunas

Colunas auxiliares (`fechamento_txt`, `nota_txt`, `historico`, `sequencia_txt`, `data_txt` e `pos`) ficam de fora. O `source_file` fica nas duas tabelas, para rastrear a origem.

In [0]:
# Seleção final das colunas das duas Silvers (descarta as colunas auxiliares)
df_silver = df_der.select(
    "protocolo", "id_uc", "canal", "assunto", "subassunto",
    "data_abertura", "data_fechamento", "mes_referencia", "aberto",
    "tempo_resolucao_horas", "qtd_tratativas", "nota_satisfacao",
    "source_file",
)
df_trat = df_trat_full.select("protocolo", "sequencia", "acao", "canal", "data_tratativa", "source_file")

df_silver.printSchema()
df_trat.printSchema()

root
 |-- protocolo: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- assunto: string (nullable = true)
 |-- subassunto: string (nullable = true)
 |-- data_abertura: timestamp (nullable = true)
 |-- data_fechamento: timestamp (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- aberto: boolean (nullable = false)
 |-- tempo_resolucao_horas: decimal(10,2) (nullable = true)
 |-- qtd_tratativas: integer (nullable = true)
 |-- nota_satisfacao: decimal(3,1) (nullable = true)
 |-- source_file: string (nullable = true)

root
 |-- protocolo: string (nullable = true)
 |-- sequencia: integer (nullable = true)
 |-- acao: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- data_tratativa: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



## 10. Checagens de qualidade

Rodam sobre o resultado, **antes** de gravar, nas duas tabelas. Cada uma protege uma decisão acima:

| Checagem | Protege |
|---|---|
| Total de chamados = 318.871 e de tratativas = 636.857 | nenhum registro perdido nem repetido (vale para esta extração) |
| `protocolo` único; `(protocolo, sequencia)` único | **cópias** (não há mais `dropDuplicates`) |
| Colunas-chave não nulas | conversões de data, número e abertura do histórico |
| `protocolo` = `AT` + 9 dígitos, `id_uc` = `UC` + 6 dígitos, `mes_referencia` = `yyyy-MM` | formato das chaves e **cabeçalho repetido** (um `protocolo = "protocolo"` não casa com o padrão) |
| `canal`, `assunto`, `subassunto` e `acao` dentro dos valores conhecidos | domínio das categorias (grafia nova falha) |
| `id_uc` existe em `silver.unidades_consumidoras` | integridade referencial com o cadastro (a Silver de UCs não tem as 200 UCs de teste) |
| Abertura entre 2022-07-01 e 2026-06-30 | período da extração |
| Abertura <= fechamento; `mes_referencia` = mês da abertura; `tempo_resolucao_horas` >= 0 | coerência das datas |
| `data_fechamento` convertida quando preenchida; nenhum chamado aberto nesta extração | conversão de data (um fechamento que não converte pareceria um chamado aberto) |
| Nota entre 1 e 5 (quando preenchida); no máximo 1 casa decimal na origem | faixa e o cast para `decimal(3,1)` |
| `qtd_tratativas` entre 1 e 3 e igual ao número de tratativas abertas | abertura do `historico` |
| Abertura <= primeira tratativa e fechamento = última tratativa | gabarito independente das datas |
| `sequencia` = posição + 1, de 1 a 3; datas das tratativas em ordem e dentro do período do chamado | integridade do histórico |
| Toda tratativa pertence a um chamado de `silver.atendimentos` | integridade entre as duas tabelas |

**Observações:**
- Cabeçalho repetido, cópias e UC inexistente **não têm tratamento** neste notebook: a exploração encontrou 0 casos. Se uma carga futura trouxer algum, a checagem correspondente falha e as Silvers não são gravadas.
- As checagens de contagem total e "nenhum chamado aberto" são proteções desta extração única. Se a Silver for reexecutada com dados novos, os valores esperados precisam ser atualizados.
- As marcadas como **nova** não foram feitas na exploração. Se alguma falhar, investigue antes de mudar a checagem.
- O gabarito do histórico, a ordem e o período das tratativas vieram da exploração (seções 7 e 9), onde valiam em 100%.

In [0]:
# Checagens de qualidade antes de gravar (condição = o que deve ser verdade em todas as linhas)
t = "silver.atendimentos"
tt = "silver.tratativas_atendimento"
dq_results.clear()   # se esta célula for reexecutada, não acumula resultados antigos

CANAIS = ["TELEFONE", "APP", "WHATSAPP", "SITE", "PRESENCIAL"]
ASSUNTOS = ["FALTA DE ENERGIA", "FATURA", "DANO ELETRICO", "LIGACAO NOVA", "GERACAO DISTRIBUIDA", "RELIGACAO"]
SUBASSUNTOS = ["VISTORIA", "INTERRUPCAO PARCIAL", "INTERRUPCAO TOTAL", "OSCILACAO", "URGENCIA", "RESSARCIMENTO",
               "APOS PAGAMENTO", "SOLICITACAO", "HOMOLOGACAO", "PRAZO", "CREDITOS", "CONTESTACAO",
               "SEGUNDA VIA", "VALOR ALTO", "PARCELAMENTO"]
ACOES = ["ABERTURA", "RETORNO AO CLIENTE", "DESPACHO", "ENCERRAMENTO", "ANALISE"]

def no_dominio(coluna, valores):
    return F.coalesce(F.col(coluna).isin(*valores), F.lit(False))   # nulo também falha

# ---------- silver.atendimentos ----------
# Volume e chaves
dq_check_contagem(t, "total de chamados = 318.871", df_silver, 318_871)
dq_check_unique(t, "protocolo único", df_silver, ["protocolo"])

# Nulos (nota_satisfacao pode ser nula: nulo de negócio, chamado não avaliado; data_fechamento nula = chamado aberto)
for coluna in ["protocolo", "id_uc", "canal", "assunto", "subassunto", "data_abertura",
               "mes_referencia", "qtd_tratativas", "source_file"]:
    dq_check(t, f"{coluna} não nulo", df_silver, F.col(coluna).isNotNull())   # source_file: nova

# Formato das chaves e domínios
dq_check(t, "protocolo no padrão AT + 9 dígitos", df_silver, F.col("protocolo").rlike("^AT[0-9]{9}$"))
dq_check(t, "id_uc no padrão UC + 6 dígitos", df_silver, F.col("id_uc").rlike("^UC[0-9]{6}$"))
dq_check(t, "mes_referencia no formato yyyy-MM", df_silver, F.col("mes_referencia").rlike("^[0-9]{4}-[0-9]{2}$"))
dq_check(t, "canal só nos 5 valores conhecidos", df_silver, no_dominio("canal", CANAIS))
dq_check(t, "assunto só nos 6 valores conhecidos", df_silver, no_dominio("assunto", ASSUNTOS))
dq_check(t, "subassunto só nos 15 valores conhecidos", df_silver, no_dominio("subassunto", SUBASSUNTOS))

# Integridade referencial com o cadastro (só checagem: não há filtro de UC neste notebook)
ids_ucs = spark.table(tabela_uc).select("id_uc").distinct()
ucs_ref = ids_ucs.withColumn("uc_no_cadastro", F.lit(True))
chamado_uc = (df_silver.join(ucs_ref, "id_uc", "left")
              .withColumn("uc_no_cadastro", F.coalesce("uc_no_cadastro", F.lit(False))))
dq_check(t, "id_uc existe em silver.unidades_consumidoras", chamado_uc, F.col("uc_no_cadastro"))

# Datas
dq_check(t, "abertura no período da extração (2022-07-01 a 2026-06-30)", df_silver,
         (F.col("data_abertura") >= F.lit("2022-07-01").cast("timestamp"))
         & (F.col("data_abertura") < F.lit("2026-07-01").cast("timestamp")))               # nova
dq_check(t, "abertura <= fechamento (quando há fechamento)", df_silver,
         F.col("data_fechamento").isNull() | (F.col("data_abertura") <= F.col("data_fechamento")))
dq_check(t, "mes_referencia = mês da data_abertura", df_silver,
         F.col("mes_referencia") == F.date_format("data_abertura", "yyyy-MM"))
dq_check(t, "data_fechamento convertida quando preenchida (e fora da sentinela)", df_der,
         F.col("fechamento_txt").isNull() | F.col("fechamento_txt").startswith("9999") | F.col("data_fechamento").isNotNull())
dq_check(t, "nenhum chamado aberto nesta extração (se falhar, ver se a conversão de data_fechamento quebrou)",
         df_silver, ~F.col("aberto"))
dq_check(t, "tempo_resolucao_horas >= 0 (chamado fechado)", df_silver,
         F.col("aberto") | (F.col("tempo_resolucao_horas") >= 0))                          # nova

# Nota (nulo de negócio é permitido)
dq_check(t, "nota entre 1 e 5 (quando preenchida)", df_silver,
         F.col("nota_satisfacao").isNull() | F.col("nota_satisfacao").between(1, 5))
dq_check(t, "nota de origem com no máximo 1 casa decimal (o cast para decimal não arredondou)", df_der,
         ~F.col("nota_txt").rlike(r"\.[0-9]{2,}$"))                                         # nova

# ---------- silver.tratativas_atendimento ----------
dq_check_contagem(tt, "total de tratativas = 636.857", df_trat, 636_857)
dq_check_unique(tt, "(protocolo, sequencia) único", df_trat, ["protocolo", "sequencia"])

for coluna in ["protocolo", "sequencia", "acao", "canal", "data_tratativa", "source_file"]:
    dq_check(tt, f"{coluna} não nulo", df_trat, F.col(coluna).isNotNull())

dq_check(tt, "protocolo no padrão AT + 9 dígitos", df_trat, F.col("protocolo").rlike("^AT[0-9]{9}$"))
dq_check(tt, "acao só nas 5 ações conhecidas", df_trat, no_dominio("acao", ACOES))
dq_check(tt, "canal só nos 5 valores conhecidos", df_trat, no_dominio("canal", CANAIS))
dq_check(tt, "sequencia entre 1 e 3", df_trat, F.col("sequencia").between(1, 3))
dq_check(tt, "sequencia = posição na lista + 1", df_trat_full, F.col("sequencia") == F.col("pos") + 1)

# Integridade entre as duas tabelas e coerência de datas
chamados_ref = df_silver.select("protocolo", "data_abertura", "data_fechamento").withColumn("tem_chamado", F.lit(True))
w_seq = Window.partitionBy("protocolo").orderBy("sequencia")
trat_chamado = (df_trat.join(chamados_ref, "protocolo", "left")
                .withColumn("tem_chamado", F.coalesce("tem_chamado", F.lit(False)))
                .withColumn("data_anterior", F.lag("data_tratativa").over(w_seq)))

dq_check(tt, "toda tratativa pertence a um chamado de silver.atendimentos", trat_chamado, F.col("tem_chamado"))
dq_check(tt, "datas das tratativas em ordem crescente", trat_chamado,
         F.col("data_anterior").isNull() | (F.col("data_tratativa") >= F.col("data_anterior")))
dq_check(tt, "tratativa não vem antes da abertura do chamado", trat_chamado,
         F.col("data_tratativa") >= F.col("data_abertura"))
dq_check(tt, "tratativa não vem depois do fechamento do chamado", trat_chamado,
         F.col("data_fechamento").isNull() | (F.col("data_tratativa") <= F.col("data_fechamento")))

resumo_trat = df_trat.groupBy("protocolo").agg(
    F.count("*").alias("n_trat"), F.min("data_tratativa").alias("primeira"), F.max("data_tratativa").alias("ultima"))
chamado_trat = df_silver.join(resumo_trat, "protocolo", "left")

dq_check(t, "todo chamado tem ao menos uma tratativa", chamado_trat, F.col("n_trat").isNotNull())
dq_check(t, "qtd_tratativas entre 1 e 3", df_silver, F.col("qtd_tratativas").between(1, 3))
dq_check(t, "qtd_tratativas = tratativas abertas em silver.tratativas_atendimento", chamado_trat,
         F.col("qtd_tratativas") == F.coalesce(F.col("n_trat"), F.lit(0)))
dq_check(t, "abertura <= primeira tratativa", chamado_trat,
         F.coalesce(F.col("data_abertura") <= F.col("primeira"), F.lit(False)))
dq_check(t, "fechamento = última tratativa (chamado fechado)", chamado_trat,
         F.col("aberto") | F.coalesce(F.col("data_fechamento") == F.col("ultima"), F.lit(False)))

[PASS] silver.atendimentos | total de chamados = 318.871 | 318.871 linhas (esperado 318.871)
[PASS] silver.atendimentos | protocolo único | 0 chaves duplicadas de 318871 linhas
[PASS] silver.atendimentos | protocolo não nulo | 0/318871 linhas falharam
[PASS] silver.atendimentos | id_uc não nulo | 0/318871 linhas falharam
[PASS] silver.atendimentos | canal não nulo | 0/318871 linhas falharam
[PASS] silver.atendimentos | assunto não nulo | 0/318871 linhas falharam
[PASS] silver.atendimentos | subassunto não nulo | 0/318871 linhas falharam
[PASS] silver.atendimentos | data_abertura não nulo | 0/318871 linhas falharam
[PASS] silver.atendimentos | mes_referencia não nulo | 0/318871 linhas falharam
[PASS] silver.atendimentos | qtd_tratativas não nulo | 0/318871 linhas falharam
[PASS] silver.atendimentos | source_file não nulo | 0/318871 linhas falharam
[PASS] silver.atendimentos | protocolo no padrão AT + 9 dígitos | 0/318871 linhas falharam
[PASS] silver.atendimentos | id_uc no padrão UC + 

## 11. Registro das checagens em `silver.dq_log`

O resultado das checagens é gravado **sempre**, mesmo quando alguma falha: o registro de uma falha é útil para investigar.

A gravação usa o modo `append` (o mesmo da aula), então **cada execução acrescenta linhas** e a tabela acumula as checagens de todas as execuções. O que diferencia uma execução da outra é a coluna `checked_at`. Ao consultar o log, filtre por `table_name` e pela execução mais recente, senão as mesmas checagens aparecem repetidas. A célula abaixo mostra só a execução atual, das duas tabelas.

In [0]:
# Persistência do dq_log (modo append, igual à aula); sempre grava, inclusive quando há falha
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{silver_schema}.dq_log")

# mostra só a execução atual (as últimas len(dq_results) linhas destas duas tabelas)
display(
    spark.table(f"{silver_schema}.dq_log")
    .filter(F.col("table_name").isin("silver.atendimentos", "silver.tratativas_atendimento"))
    .orderBy(F.col("checked_at").desc())
    .limit(len(dq_results))
)

table_name,check_name,total_rows,failed_rows,passed,checked_at
silver.atendimentos,fechamento = última tratativa (chamado fechado),318871,0,true,2026-10-26T20:54:50.097Z
silver.atendimentos,abertura <= primeira tratativa,318871,0,true,2026-10-26T20:54:47.753Z
silver.atendimentos,qtd_tratativas = tratativas abertas em silver.tratativas_atendimento,318871,0,true,2026-10-26T20:54:45.254Z
silver.atendimentos,qtd_tratativas entre 1 e 3,318871,0,true,2026-10-26T20:54:43.367Z
silver.atendimentos,todo chamado tem ao menos uma tratativa,318871,0,true,2026-10-26T20:54:42.517Z
silver.tratativas_atendimento,tratativa não vem depois do fechamento do chamado,636857,0,true,2026-10-26T20:54:40.411Z
silver.tratativas_atendimento,tratativa não vem antes da abertura do chamado,636857,0,true,2026-10-26T20:54:36.571Z
silver.tratativas_atendimento,datas das tratativas em ordem crescente,636857,0,true,2026-10-26T20:54:32.784Z
silver.tratativas_atendimento,toda tratativa pertence a um chamado de silver.atendimentos,636857,0,true,2026-10-26T20:54:30.041Z
silver.tratativas_atendimento,sequencia = posição na lista + 1,636857,0,true,2026-10-26T20:54:27.066Z


## 12. Gravação das Silvers, só se todas as checagens passarem

A Gold lê as Silvers. Por isso, se **qualquer** checagem falhar, **nenhuma** das duas tabelas é gravada: o notebook imprime quais checagens falharam e é interrompido com erro (num Workflow, isso marca a execução como falha e impede que dados com problema sigam para as camadas seguintes). Se as tabelas já existiam de uma execução anterior, permanecem como estavam.

As duas são gravadas juntas para que `silver.atendimentos` e `silver.tratativas_atendimento` nunca fiquem de versões diferentes.

In [0]:
# Gravação das Silvers: só acontece se todas as checagens passaram
falhas = [r for r in dq_results if not r.passed]

if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_silver} e {tabela_silver_trat} NÃO foram gravadas:")
    for r in falhas:
        print(f"  - {r.table_name} | {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"Silvers de atendimentos não gravadas: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

for df_saida, destino in [(df_silver, tabela_silver), (df_trat, tabela_silver_trat)]:
    (
        df_saida.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(destino)
    )
    print(f"{destino} gravada: {n_br(spark.table(destino).count())} linhas")   # esperado: 318.871 e 636.857
print(f"Todas as {len(dq_results)} checagens passaram.")

v_energy.silver.atendimentos gravada: 318.871 linhas
v_energy.silver.tratativas_atendimento gravada: 636.857 linhas
Todas as 48 checagens passaram.


## 13. Bronze × Silver, com as mesmas linhas

Para cada caso, os `protocolo` são escolhidos e depois mostrados na Bronze e nas duas Silvers, então dá para ver o **texto original** (`2025-07-14 10:30:00`, `3.9`, a lista do `historico`) ao lado do valor tratado. Não há casos de cabeçalho repetido, cópia nem UC inexistente para mostrar: a Silver tem os mesmos 318.871 chamados da Bronze, e a célula imprime as duas contagens.

Os casos mostrados:
1. chamado **sem nota** (nulo de negócio, mantido nulo);
2. chamado **com nota** e 3 tratativas (nota convertida; histórico aberto em 3 linhas);
3. chamado do **pico de julho de 2025** (mantido como veio).

In [0]:
# Bronze × Silver: os MESMOS protocolos nas três tabelas, um grupo de exemplos por caso
silver_a = spark.table(tabela_silver)
silver_t = spark.table(tabela_silver_trat)
colunas_bronze = ["protocolo", "id_uc", "canal", "assunto", "subassunto",
                  "data_abertura", "data_fechamento", "nota_satisfacao", "historico"]

def comparar(titulo, protocolos_df, n=2):
    ids = [r["protocolo"] for r in protocolos_df.select("protocolo").limit(n).collect()]
    print(f"=== {titulo} | protocolos: {ids}")
    print("Bronze (texto original):")
    display(bronze.filter(F.col("protocolo").isin(ids)).select(*colunas_bronze).orderBy("protocolo"))
    print("Silver · atendimentos (tratado):")
    display(silver_a.filter(F.col("protocolo").isin(ids)).orderBy("protocolo"))
    print("Silver · tratativas_atendimento (histórico aberto):")
    display(silver_t.filter(F.col("protocolo").isin(ids)).orderBy("protocolo", "sequencia"))

# Casos sem exemplo (esperado: 0 em todos)
print("chamados na Bronze:", n_br(n_bronze), "| na Silver:", n_br(silver_a.count()))   # esperado: 318.871 nas duas

# 1. Nota nula (nulo de negócio)
comparar("Chamado sem nota (mantida nula)", silver_a.filter(F.col("nota_satisfacao").isNull()))

# 2. Nota preenchida e 3 tratativas
comparar("Chamado com nota e 3 tratativas",
         silver_a.filter(F.col("nota_satisfacao").isNotNull() & (F.col("qtd_tratativas") == 3)))

# 3. Pico de julho de 2025
comparar("Chamado do pico de julho de 2025", silver_a.filter(F.col("mes_referencia") == "2025-07"))

chamados na Bronze: 318.871 | na Silver: 318.871
=== Chamado sem nota (mantida nula) | protocolos: ['AT000000011', 'AT000000020']
Bronze (texto original):


protocolo,id_uc,canal,assunto,subassunto,data_abertura,data_fechamento,nota_satisfacao,historico
AT000000011,UC142212,WHATSAPP,DANO ELETRICO,VISTORIA,2022-07-21 02:25:00,2022-07-22 14:25:00,null,"List(List(DESPACHO, WHATSAPP, 2022-07-21T14:25:00, 1), List(ANALISE, WHATSAPP, 2022-07-22T02:25:00, 2), List(RETORNO AO CLIENTE, WHATSAPP, 2022-07-22T14:25:00, 3))"
AT000000020,UC139580,TELEFONE,FALTA DE ENERGIA,INTERRUPCAO PARCIAL,2022-07-16 02:54:00,2022-07-16 08:54:00,null,"List(List(RETORNO AO CLIENTE, TELEFONE, 2022-07-16T08:54:00, 1))"


Silver · atendimentos (tratado):


protocolo,id_uc,canal,assunto,subassunto,data_abertura,data_fechamento,mes_referencia,aberto,tempo_resolucao_horas,qtd_tratativas,nota_satisfacao,source_file
AT000000011,UC142212,WHATSAPP,DANO ELETRICO,VISTORIA,2022-07-21T02:25:00.000Z,2022-07-22T14:25:00.000Z,2022-07,false,36.00,3,null,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json
AT000000020,UC139580,TELEFONE,FALTA DE ENERGIA,INTERRUPCAO PARCIAL,2022-07-16T02:54:00.000Z,2022-07-16T08:54:00.000Z,2022-07,false,6.00,1,null,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json


Silver · tratativas_atendimento (histórico aberto):


...

protocolo,id_uc,canal,assunto,subassunto,data_abertura,data_fechamento,nota_satisfacao,historico
AT000230048,UC143748,APP,LIGACAO NOVA,VISTORIA,2025-07-24 22:45:00,2025-08-23 07:45:00,null,"List(List(ABERTURA, APP, 2025-08-08T14:45:00, 1), List(RETORNO AO CLIENTE, APP, 2025-08-23T07:45:00, 2))"
AT000230049,UC118400,APP,FALTA DE ENERGIA,INTERRUPCAO PARCIAL,2025-07-25 06:58:00,2025-08-23 07:58:00,2.7,"List(List(ENCERRAMENTO, APP, 2025-08-03T22:58:00, 1), List(ANALISE, APP, 2025-08-13T14:58:00, 2), List(ABERTURA, APP, 2025-08-23T07:58:00, 3))"


Silver · atendimentos (tratado):


protocolo,id_uc,canal,assunto,subassunto,data_abertura,data_fechamento,mes_referencia,aberto,tempo_resolucao_horas,qtd_tratativas,nota_satisfacao,source_file
AT000230048,UC143748,APP,LIGACAO NOVA,VISTORIA,2025-07-24T22:45:00.000Z,2025-08-23T07:45:00.000Z,2025-07,false,705.00,2,null,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json
AT000230049,UC118400,APP,FALTA DE ENERGIA,INTERRUPCAO PARCIAL,2025-07-25T06:58:00.000Z,2025-08-23T07:58:00.000Z,2025-07,false,697.00,3,2.7,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json


Silver · tratativas_atendimento (histórico aberto):


protocolo,sequencia,acao,canal,data_tratativa,source_file
AT000230048,1,ABERTURA,APP,2025-08-08T14:45:00.000Z,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json
AT000230048,2,RETORNO AO CLIENTE,APP,2025-08-23T07:45:00.000Z,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json
AT000230049,1,ENCERRAMENTO,APP,2025-08-03T22:58:00.000Z,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json
AT000230049,2,ANALISE,APP,2025-08-13T14:58:00.000Z,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json
AT000230049,3,ABERTURA,APP,2025-08-23T07:58:00.000Z,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json


## 14. Conferência final contra a exploração

Os números abaixo vêm de `exploracao_atendimentos.ipynb`.

In [0]:
# Conferência final (os valores esperados vêm da exploração)
print("chamados:", n_br(silver_a.count()), "| tratativas:", n_br(silver_t.count()))   # esperado: 318.871 | 636.857
print("UCs com ao menos um chamado:", n_br(silver_a.select("id_uc").distinct().count()), "(de 45.000)")   # esperado: 44.940

p0 = silver_a.filter(F.col("mes_referencia").between("2022-07", "2023-06")).count()   # esperado: 76.561
p1 = silver_a.filter(F.col("mes_referencia").between("2025-07", "2026-06")).count()   # esperado: 88.823
print(f"primeiro ano: {n_br(p0)} | últimos 12 meses: {n_br(p1)} | variação: {(p1 / p0 - 1) * 100:.1f}%")   # esperado: ~+16%

# chamados por mês: ~6.400 por mês (de 6.237 a 6.624), exceto 2025-07 = 18.061
silver_a.groupBy("mes_referencia").count().orderBy("mes_referencia").show(60)

# nota: 309.305 preenchidas, 9.566 nulas, de 1,3 a 5,0
silver_a.agg(F.count("nota_satisfacao").alias("com_nota"),
             F.sum(F.col("nota_satisfacao").isNull().cast("int")).alias("sem_nota"),
             F.min("nota_satisfacao").alias("min"), F.max("nota_satisfacao").alias("max")).show()

# julho de 2025 x demais meses (esperado: ~361 h contra ~120 h até fechar; nota ~3,11 contra ~4,10)
(silver_a.withColumn("grupo", F.when(F.col("mes_referencia") == "2025-07", "2025-07").otherwise("demais"))
         .groupBy("grupo")
         .agg(F.count("*").alias("chamados"),
              F.round(F.avg(F.col("tempo_resolucao_horas").cast("double")), 1).alias("horas_ate_fechar"),
              F.round(F.avg(F.col("nota_satisfacao").cast("double")), 2).alias("nota_media"))
         .show())

chamados: 318.871 | tratativas: 636.857
UCs com ao menos um chamado: 44.940 (de 45.000)
primeiro ano: 76.561 | últimos 12 meses: 88.823 | variação: 16.0%
+--------------+-----+
|mes_referencia|count|
+--------------+-----+
|       2022-07| 6322|
|       2022-08| 6474|
|       2022-09| 6309|
|       2022-10| 6326|
|       2022-11| 6467|
|       2022-12| 6377|
|       2023-01| 6323|
|       2023-02| 6448|
|       2023-03| 6315|
|       2023-04| 6482|
|       2023-05| 6356|
|       2023-06| 6362|
|       2023-07| 6449|
|       2023-08| 6457|
|       2023-09| 6320|
|       2023-10| 6555|
|       2023-11| 6348|
|       2023-12| 6513|
|       2024-01| 6325|
|       2024-02| 6425|
|       2024-03| 6494|
|       2024-04| 6392|
|       2024-05| 6414|
|       2024-06| 6382|
|       2024-07| 6456|
|       2024-08| 6304|
|       2024-09| 6348|
|       2024-10| 6470|
|       2024-11| 6342|
|       2024-12| 6398|
|       2025-01| 6422|
|       2025-02| 6471|
|       2025-03| 6312|
|       2025-04| 6